# 🚀 Antiface 1-Click Free GPU Worker (Nvidia T4 / P100)
Outsource heavy computing, AI generation, and processing from **@Antiface00bot** on Telegram directly to this free Google Colab / Kaggle GPU.

### ⚡ Instructions:
1. Ensure GPU is enabled: **Runtime ➔ Change runtime type ➔ T4 GPU**.
2. Click **Runtime ➔ Run all** (or press `Ctrl+F9`).
3. That is it! The notebook will automatically register its tunnel with your bot and send a confirmation to your Telegram!

In [ ]:
# 1. Install dependencies
!pip install -q fastapi uvicorn pydantic requests torch
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import os, sys, time, re, subprocess, requests, torch
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from threading import Thread

print('\n====================================')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU Device:', gpu_name)
vram = f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB' if torch.cuda.is_available() else 'N/A'
print('VRAM Available:', vram)
print('====================================\n')

app = FastAPI()

class TaskRequest(BaseModel):
    prompt: str
    action: str = 'execute'

@app.get('/health')
def health():
    return {
        'status': 'healthy',
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'torch_version': torch.__version__
    }

@app.post('/run')
def run_task(req: TaskRequest):
    try:
        device_str = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
        result = f'Successfully processed on {device_str} (VRAM: {vram}): {req.prompt}'
        return {'success': True, 'result': result}
    except Exception as e:
        return {'success': False, 'error': str(e)}

def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='error')

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

# Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stderr=subprocess.PIPE, text=True)
tunnel_url = None

for line in tunnel_proc.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

print('🌐 Cloudflare Tunnel Established:', tunnel_url)

# Auto-register with your Antiface Cloudflare Worker
WORKER_ENDPOINT = 'https://antiface-bot.prishatsharma8.workers.dev/register-gpu'
try:
    resp = requests.post(WORKER_ENDPOINT, json={'url': tunnel_url}, timeout=10)
    print('✅ Registered with @Antiface00bot:', resp.json())
    print('\n🎉 GPU WORKER IS FULLY ACTIVE! Send /gpu <prompt> on Telegram now.')
except Exception as e:
    print('Registration failed:', e)

# Keep process alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('Stopping worker.')
